# SAFE binary v7 — 금전 사기 커버리지 보강 (합성+LLM검수 → 재학습 → fresh blind v9)

v6 파이프라인 ⊇ + 앞단에 **사기 데이터 생성**(Gemini 합성 → LLM 라벨 검수 → blind 누수 제거).
GPU 런타임에서 위→아래로 실행. GEMINI_API_KEY와 data_bundle.zip이 필요하다.
fresh blind는 v9(사기 슬라이스 + 정상 금전 경계 반례). dev 회귀셋 = 실holdout + blind v1~v8.

In [ ]:
# 1. GPU 및 저장소 확인
import subprocess, sys, json, hashlib, re, shutil, collections
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다.'
print(torch.cuda.get_device_name(0))
REPO=Path('/content/thisabled-ai')
BRANCH='feature/safe-scam-augmentation'
REMOTE='https://github.com/threeGuineas/thisabled-ai.git'
if REPO.exists():
    subprocess.run(['git','pull','--ff-only','origin',BRANCH],cwd=REPO,check=True)
else:
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REMOTE,str(REPO)],check=True)
assert (REPO/'.git').exists(), f'저장소 준비 실패: {REPO}'
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],cwd=REPO,check=True)
sys.path.insert(0,str(REPO))

In [ ]:
# 2. data_bundle.zip 업로드 — VS Code/Jupyter widget
import io, zipfile, ipywidgets as widgets
from IPython.display import display
uploader=widgets.FileUpload(accept='.zip',multiple=False,description='data_bundle.zip 선택')
def on_upload(change):
    value=uploader.value
    if not value: return
    item=next(iter(value.values())) if isinstance(value,dict) else value[0]
    with zipfile.ZipFile(io.BytesIO(bytes(item['content']))) as z:
        required={'data/eval/aihub_train.jsonl','data/eval/aihub_real_holdout.jsonl','data/eval/beep_real_holdout.jsonl'}
        missing=sorted(required-set(z.namelist())); assert not missing, missing
        z.extractall(REPO)
    print('업로드 완료')
uploader.observe(on_upload,names='value'); display(uploader)

In [ ]:
# 3. 금전 사기 학습 데이터 생성 (Gemini 합성 → LLM 라벨 검수 → blind v1~v9 누수 제거)
import os
from getpass import getpass
os.environ['GEMINI_API_KEY']=getpass('GEMINI_API_KEY: ')  # 노트북에 키 안 남김
cmd=[sys.executable,'scripts/build_scam_dataset.py','--per-subtype','40']
for i in range(1,10): cmd+=['--forbidden', f'tests/fixtures/safe_blind_v{i}.jsonl']
subprocess.run(cmd,cwd=REPO,check=True)
del os.environ['GEMINI_API_KEY']
scam=[json.loads(x) for x in (REPO/'data/synthetic/scam/train.jsonl').read_text().splitlines() if x]
print('scam train',len(scam),'| label',dict(collections.Counter(x['label'] for x in scam)))
assert len(scam)>=50, '사기 데이터가 너무 적음 — 프롬프트/키/검수 확인'

In [ ]:
# 4. 데이터 빌드 + 외부 어댑터 + 하드케이스 v5 + 누수 가드 (dev=blind v1~v8 / fresh=v9)
required=[REPO/'data/eval/aihub_train.jsonl',REPO/'data/eval/aihub_real_holdout.jsonl',REPO/'data/eval/beep_real_holdout.jsonl']
assert all(p.exists() for p in required),[str(p) for p in required if not p.exists()]
BLIND_V9=REPO/'tests/fixtures/safe_blind_v9.jsonl'
assert BLIND_V9.exists(), 'fresh blind v9 없음 — tests/fixtures/safe_blind_v9.jsonl(40행)'
steps=[
    [sys.executable,'scripts/download_seed_datasets.py'],
    [sys.executable,'scripts/build_processed_dataset.py'],
    [sys.executable,'scripts/build_final_dataset.py','--synth-repeat','1','--include-aihub-train'],
    [sys.executable,'scripts/build_safe_hardcase_dataset.py','--include-v5',
     '--output','data/synthetic/safe_hardcases_v5/train.jsonl',
     '--forbidden','tests/fixtures/safe_blind_v7.jsonl',
     '--forbidden','tests/fixtures/safe_blind_v8.jsonl',
     '--forbidden','tests/fixtures/safe_blind_v9.jsonl'],
    [sys.executable,'scripts/adapt_external_datasets.py'],
]
for cmd in steps: subprocess.run(cmd,cwd=REPO,check=True)
import pandas as pd
from src.data.dedup import find_duplicate_indices
train=pd.read_parquet(REPO/'data/processed/train.parquet')
def norm(x): return re.sub(r'[^0-9a-z가-힣]+','',str(x).lower())
train_norm={norm(t) for t in train['text']}
extra_paths=['data/synthetic/dktc.jsonl','data/synthetic/kmhas.jsonl','data/synthetic/apeach.jsonl',
             'data/synthetic/safe_hardcases_v5/train.jsonl','data/synthetic/scam/train.jsonl']
extra=[]
for p in extra_paths: extra+=[json.loads(x) for x in (REPO/p).read_text().splitlines() if x]
extra_texts=[x['text'] for x in extra]; extra_norm={norm(t) for t in extra_texts}
blindv9=[json.loads(x) for x in BLIND_V9.read_text().splitlines() if x]; bt=[x['text'] for x in blindv9]
for i in range(1,9):
    blind=[json.loads(x) for x in (REPO/'tests/fixtures'/f'safe_blind_v{i}.jsonl').read_text().splitlines() if x]
    assert not ({norm(x['text']) for x in blind}&extra_norm), f'extra leak vs safe_blind_v{i}'
assert not ({norm(t) for t in bt}&extra_norm), 'blind v9 exact leak vs extra'
assert not ({norm(t) for t in bt}&train_norm), 'blind v9 exact leak vs train'
assert not find_duplicate_indices(extra_texts, bt, threshold=0.8), 'blind v9 near-dup vs extra'
assert not find_duplicate_indices(list(train['text']), bt, threshold=0.8), 'blind v9 near-dup vs train'
print('base train',len(train),'| extra',len(extra),
      '| extra label',pd.Series([x.get('label') for x in extra]).value_counts(dropna=False).to_dict())
print('blind v9 clean vs train/extra: OK')

In [ ]:
# 5. 개발 평가 함수 — dev 회귀셋 = 실 holdout + 소비된 blind v1~v8. v9는 여기서 읽지 않는다. 규칙보조 OFF.
import numpy as np, yaml
from sklearn.metrics import confusion_matrix
from transformers import AutoTokenizer,AutoModelForSequenceClassification
real=[]
for name in ['aihub_real_holdout.jsonl','beep_real_holdout.jsonl']:
    real += [json.loads(x) for x in (REPO/'data/eval'/name).read_text().splitlines() if x]
dev=[]
for i in range(1,9):
    dev += [json.loads(x) for x in (REPO/'tests/fixtures'/f'safe_blind_v{i}.jsonl').read_text().splitlines() if x]
groom=[]
for split in ['val','test']: groom += [json.loads(x) for x in (REPO/f'data/synthetic/emergency/3a/{split}.jsonl').read_text().splitlines() if x]
def load_predict(path):
    tok=AutoTokenizer.from_pretrained(path); model=AutoModelForSequenceClassification.from_pretrained(path).cuda().eval(); assert model.config.num_labels==2
    def predict(texts,batch=128):
        out=[]
        with torch.inference_mode():
            for i in range(0,len(texts),batch):
                enc=tok(texts[i:i+batch],padding=True,truncation=True,max_length=128,return_tensors='pt').to('cuda')
                out.extend(torch.softmax(model(**enc).logits,dim=-1)[:,1].cpu().tolist())
        return np.array(out)
    return model,predict
sl=lambda key: [i for i,x in enumerate(dev) if str(x['slice'])==key]
fraud=sl('fraud_credentials'); coercive=sl('coercive_control'); ext=sl('digital_extortion')
grooming=[i for i,x in enumerate(dev) if str(x['slice']).startswith('grooming')]
recon=sl('routine_recon')
def evaluate_dev(path):
    model,predict=load_predict(path)
    yr=np.array([int(int(x['label'])>0) for x in real]); pr=predict([x['text'] for x in real])
    yd=np.array([x['label'] for x in dev]); pdv=predict([x['text'] for x in dev])
    pg=predict([x['text'] for x in groom])
    candidates=[]
    for adult in np.arange(.40,.86,.01):
        minor=max(.35,round(float(adult)-.16,2))
        rpred=(pr>=adult)
        dthr=np.array([minor if x['receiver_is_minor'] else adult for x in dev]); dpred=(pdv>=dthr)
        tn,fp,fn,tp=confusion_matrix(yr,rpred,labels=[0,1]).ravel(); dtn,dfp,dfn,dtp=confusion_matrix(yd,dpred,labels=[0,1]).ravel()
        row={'adult':round(float(adult),2),'minor':minor,'real_recall':tp/(tp+fn),'real_specificity':tn/(tn+fp),
             'dev_recall':dtp/(dtp+dfn),'dev_specificity':dtn/(dtn+dfp),
             'fraud_recall':float(dpred[fraud].mean()) if fraud else 1.0,'coercive_recall':float(dpred[coercive].mean()) if coercive else 1.0,
             'grooming_recall':float(dpred[grooming].mean()) if grooming else 1.0,'extortion_recall':float(dpred[ext].mean()) if ext else 1.0,
             'recon_recall':float(dpred[recon].mean()) if recon else 1.0,'synthetic_grooming_recall':float((pg>=minor).mean())}
        row['pass']=all([row['real_recall']>=.80,row['real_specificity']>=.80,row['dev_recall']>=.85,row['dev_specificity']>=.90,
                         row['fraud_recall']>=.80,row['coercive_recall']>=.80,row['grooming_recall']>=.80,row['extortion_recall']>=.80])
        candidates.append(row)
    passing=[x for x in candidates if x['pass']]
    best=max(passing,key=lambda x:(x['real_specificity'],x['dev_specificity'])) if passing else max(candidates,key=lambda x:(min(x['dev_recall'],x['dev_specificity'],x['fraud_recall'],x['coercive_recall']),x['real_specificity']))
    del model; torch.cuda.empty_cache(); return best

In [ ]:
# 6. v7 repeat 1→3 재학습. 개발 게이트 통과 시 즉시 중단
ATTEMPTS=[]; SELECTED=None
base=yaml.safe_load((REPO/'configs/module1_binary_hardcases_v7.yaml').read_text())
for repeat in [1,2,3]:
    cfg=json.loads(json.dumps(base)); name=f'module1_binary_hardcases_v7_r{repeat}'
    cfg['data']['extra_train_repeat']=repeat; cfg['model']['checkpoint_dir']=f'models/checkpoints/{name}'; cfg['paths']['checkpoint_dir']=f'models/checkpoints/{name}'
    temp=Path(f'/content/{name}.yaml'); temp.write_text(yaml.safe_dump(cfg,allow_unicode=True,sort_keys=False))
    subprocess.run([sys.executable,'scripts/train_module1.py','--config',str(temp)],cwd=REPO,check=True)
    ckpt=REPO/cfg['model']['checkpoint_dir']; result=evaluate_dev(ckpt); result.update({'repeat':repeat,'checkpoint':str(ckpt)}); ATTEMPTS.append(result); print(json.dumps(result,ensure_ascii=False,indent=2))
    if result['pass']: SELECTED=result; break
report=REPO/'reports/validation_reports/module1_binary_hardcases_v7/dev_attempts.json'; report.parent.mkdir(parents=True,exist_ok=True); report.write_text(json.dumps(ATTEMPTS,ensure_ascii=False,indent=2))
assert SELECTED is not None, '3회 모두 개발 게이트 실패 — blind v9 실행 및 업로드 금지'
print('SELECTED',SELECTED)

In [ ]:
# 7. 후보 고정 후 fresh blind v9 최초 1회 평가 (규칙보조 OFF). 사기 슬라이스·정상 금전 경계 감시
blind_path=REPO/'tests/fixtures/safe_blind_v9.jsonl'; before=hashlib.sha256(blind_path.read_bytes()).hexdigest()
out=REPO/'artifacts/safe_blind_v9_results.json'
subprocess.run([sys.executable,'scripts/evaluate_safe_blind.py','--model',SELECTED['checkpoint'],'--data',str(blind_path),
                '--adult-threshold',str(SELECTED['adult']),'--minor-threshold',str(SELECTED['minor']),'--no-rule-assist','--output',str(out)],cwd=REPO,check=True)
assert hashlib.sha256(blind_path.read_bytes()).hexdigest()==before, 'blind v9 원문이 변경됨 — 무효'
BLIND=json.loads(out.read_text()); m=BLIND['overall']
by=BLIND['by_slice']
risk_slices={s:by[s]['risk_recall'] for s in by if by[s].get('risk_recall') is not None}
scam_slices={s:v for s,v in risk_slices.items() if s.startswith('scam')}
min_slice=min(risk_slices.values()) if risk_slices else 1.0
BLIND_PASS=bool(m['risk_recall']>=.80 and m['specificity']>=.90 and min_slice>=.75)
print('overall',m)
print('scam 슬라이스 recall',scam_slices)
print('정상 금전 경계(benign_money) specificity',by.get('benign_money',{}).get('specificity'))
print({'blind_pass':BLIND_PASS,'min_slice_recall':round(min_slice,3),'sha256':before})
assert BLIND_PASS, 'blind v9 실패 — 업로드 금지, v9는 회귀셋으로 격하하고 다음 라운드 준비'

In [ ]:
# 8. 오류 케이스 확인 (FN=놓친 위험 / FP=과플래그)
res=json.loads((REPO/'artifacts/safe_blind_v9_results.json').read_text())
for e in res.get('errors',[]):
    kind='FN' if e['label']==1 else 'FP'
    print(kind, e['slice'], round(e.get('risk_prob',0),4), '|', e['text'])

In [ ]:
# 9. 명시적으로 켠 경우에만 HF 업로드 (추론 파일만; 학습 상태 자동 제외)
UPLOAD_TO_HF=False
if UPLOAD_TO_HF:
    from getpass import getpass
    from huggingface_hub import login,upload_folder
    token=getpass('HF write token: '); login(token=token,add_to_git_credential=False); del token
    url=upload_folder(repo_id='soyuncj/thisabled-safety-kcelectra',folder_path=SELECTED['checkpoint'],
        commit_message=f"retrain scam-augmented v7 r{SELECTED['repeat']} blind-v9 approved",
        ignore_patterns=['checkpoint-*','optimizer*','scheduler*','trainer_state*','rng_state*','training_args*'])
    print('HF_COMMIT_URL:',url)
    print('업로드 후: 새 커밋 SHA를 SAFE_MODEL_REVISION으로 서빙에 반영하고 /health revision 확인.')
else:
    print('검증 완료. 업로드는 비활성 상태입니다.')